# Notebook 04 · Same-Source Validation

Every predicted raster scored against **the product it was trained on** — CLMS
or GHS-BUILT-S in Milan, GHSL in Hanoi and HCMC — at that run's own held-out
test points.

This measures **agreement with the training target, not correctness**. A map
can match its training product closely and still be wrong about the ground.
Notebook 05 answers that separate question against the EarthLabel
photo-interpreted plots, and the two are never merged or compared directly.

**Continuous metrics only** — RMSE, MAE, bias and R² on the raw predicted
percentage, with no threshold and no binning. The confusion-matrix techniques
notebook 05 applies to the photo-interpreted plots are not repeated here:
binning a prediction against the very product it was fitted to adds nothing
the continuous error does not already say, and a high κ against one's own
training target would invite being read as accuracy.

It reads existing outputs only: no Earth Engine, no retraining, no new
sampling. Each run already wrote its held-out test points with the training
target value attached, so the reference here is exactly what the model was
fitted against.

---

## Setup

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import rasterio

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

REPO    = Path.cwd()
OUT     = REPO / "output"
RESULTS = OUT / "validation_samesource"
RESULTS.mkdir(parents=True, exist_ok=True)

print("results ->", RESULTS)

results -> D:\06_Polimi\2025-2026\IMD-Mapping\output\validation_samesource


---

## The runs, and the target each is scored against

In [2]:
# Every predicted raster, with the test points carrying its OWN training
# target. The pairing is the whole point: a CLMS-trained map is scored against
# CLMS, a GHSL-trained map against GHSL. Mixing them would not be same-source.
#
# `IMD` in each test-point file is the training-target value sampled at that
# point, written by the notebook that produced the run.

MILAN = OUT / "milan"
VIET  = OUT / "transfer_vietnam"

RUNS = []

# Milan -- four predictor sets x two training targets
for target, ras in (("clms", "IMD_predicted_RF_S2_Milan.tif"),
                    ("ghsl", "GHSL_predicted_RF_S2_Milan.tif")):
    for predictor in ("median", "stack", "percentile"):
        RUNS.append(dict(
            city="Milan", target=target.upper(), predictor=predictor,
            raster=MILAN / target / predictor / ras,
            points=MILAN / target / predictor / "spatial_test_pts.gpkg"))

for target, ras in (("clms", "IMD_predicted_RF_spatialCV2_Milan.tif"),
                    ("ghsl", "GHSL_predicted_RF_spatialCV2_Milan.tif")):
    RUNS.append(dict(
        city="Milan", target=target.upper(), predictor="embedding",
        raster=MILAN / target / "embedding" / ras,
        points=MILAN / target / "embedding" / "spatial_test_pts.gpkg"))

# Vietnam -- two predictors x two scenarios, all against GHSL
for city in ("Hanoi", "HCMC"):
    for predictor, suffix in (("embedding", ""), ("median", "_S2median")):
        for scenario in ("zeroshot", "localrf"):
            RUNS.append(dict(
                city=city, target="GHSL",
                predictor=f"{predictor}_{scenario}",
                raster=VIET / city.lower() / predictor
                       / f"IMD_{city}_10m_{scenario}{suffix}.tif",
                points=VIET / city.lower() / predictor
                       / f"samples_{city}_test.gpkg"))

print(f"{len(RUNS)} runs registered\n")
missing = 0
for r in RUNS:
    ok_r = "ok " if r["raster"].exists() else "MISS"
    ok_p = "ok " if r["points"].exists() else "MISS"
    if "MISS" in (ok_r, ok_p):
        missing += 1
        print(f"  {ok_r} raster  {ok_p} points   {r['city']:6s} "
              f"{r['target']:5s} {r['predictor']}")
print(f"{len(RUNS) - missing} / {len(RUNS)} runs have both raster and points")
assert missing == 0, f"{missing} run(s) incomplete -- see above"

16 runs registered

16 / 16 runs have both raster and points


---

## Sampling the predicted rasters at the held-out points

In [3]:
def sample_raster(raster_path, gdf):
    """Predicted value at each point centre, reprojected to the raster's CRS."""
    with rasterio.open(raster_path) as src:
        pts = gdf.to_crs(src.crs)
        coords = [(p.x, p.y) for p in pts.geometry]
        vals = np.array([v[0] for v in src.sample(coords)], dtype="float64")
        nodata = src.nodata
    if nodata is not None:
        vals[vals == nodata] = np.nan
    return vals


# The target column is named after the product: the CLMS runs write `IMD`,
# the GHSL runs write `GHSL`. Resolved rather than assumed, so a run whose
# points carry neither fails loudly instead of being scored against the
# wrong column.
TARGET_COL = {"CLMS": "IMD", "GHSL": "GHSL"}


def load_pair(run):
    """Observed (training target) and predicted % at the held-out test points."""
    gdf = gpd.read_file(run["points"])
    col = TARGET_COL[run["target"]]
    if col not in gdf.columns:
        # Vietnam transfer samples are GHSL-labelled but written by the
        # transfer notebooks, which kept the generic `IMD` name.
        col = "IMD" if "IMD" in gdf.columns else col
    if col not in gdf.columns:
        raise KeyError(
            f"{run['points']} has no target column; "
            f"expected {TARGET_COL[run['target']]!r} or 'IMD', "
            f"found {[c for c in gdf.columns if not c.startswith('A')]}")
    obs = gdf[col].to_numpy(dtype="float64")
    pred = sample_raster(run["raster"], gdf)
    keep = np.isfinite(obs) & np.isfinite(pred)
    return np.clip(obs[keep], 0, 100), np.clip(pred[keep], 0, 100)


print("sampling every run ...")
DATA = {}
for r in RUNS:
    key = (r["city"], r["target"], r["predictor"])
    DATA[key] = load_pair(r)
    n_all = len(gpd.read_file(r["points"]))
    n_ok = len(DATA[key][0])
    flag = "" if n_ok == n_all else f"   ({n_all - n_ok} dropped: nodata)"
    print(f"  {r['city']:6s} {r['target']:5s} {r['predictor']:20s} n={n_ok}{flag}")

sampling every run ...


  Milan  CLMS  median               n=1014


  Milan  CLMS  stack                n=1014


  Milan  CLMS  percentile           n=1014


  Milan  GHSL  median               n=998


...

  Hanoi  GHSL  embedding_zeroshot   n=895


  Hanoi  GHSL  embedding_localrf    n=895
  Hanoi  GHSL  median_zeroshot      n=895


  Hanoi  GHSL  median_localrf       n=895
  HCMC   GHSL  embedding_zeroshot   n=887


  HCMC   GHSL  embedding_localrf    n=887


  HCMC   GHSL  median_zeroshot      n=887
  HCMC   GHSL  median_localrf       n=887


---

## Continuous agreement with the training target

In [4]:
# The predicted percentage against the training-target percentage, exactly as
# both are stored: no threshold, no binning.

def continuous_metrics(obs, pred):
    # Bias is OBSERVED MINUS PREDICTED throughout this project, so a positive
    # value means the map under-predicts. RMSE and MAE are symmetric in the
    # residual and so are unaffected by which way round it is taken; bias is
    # not, and a silent flip here would invert every conclusion drawn from it.
    resid = obs - pred
    ss_res = float(np.sum(resid ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    return dict(
        n=len(obs),
        RMSE_pp=float(np.sqrt(np.mean(resid ** 2))),
        MAE_pp=float(np.mean(np.abs(resid))),
        bias_pp=float(np.mean(resid)),
        R2=1 - ss_res / ss_tot if ss_tot else np.nan,
        pearson_r=float(np.corrcoef(obs, pred)[0, 1]) if len(obs) > 1 else np.nan,
    )


rows = [dict(city=c, target=t, predictor=p,
             **continuous_metrics(*DATA[(c, t, p)]))
        for (c, t, p) in DATA]
metrics = (pd.DataFrame(rows)
           .sort_values(["city", "target", "R2"], ascending=[True, True, False]))
metrics.to_csv(RESULTS / "samesource_metrics.csv", index=False)

for city in ("Milan", "Hanoi", "HCMC"):
    sub = metrics[metrics.city == city]
    if len(sub):
        print(f"\n===== {city} — same-source validation =====")
        print(sub.drop(columns="city").to_string(index=False))


===== Milan — same-source validation =====
target  predictor    n  RMSE_pp  MAE_pp  bias_pp    R2  pearson_r
  CLMS percentile 1014    9.462   6.389   -0.104 0.927      0.963
  CLMS      stack 1014   10.864   7.641   -0.169 0.904      0.951
  CLMS     median 1014   11.285   7.820    0.046 0.896      0.947
  CLMS  embedding 1014   14.123  10.675    0.624 0.837      0.917
  GHSL percentile  998   18.206  13.523    0.341 0.737      0.859
  GHSL      stack  998   18.288  13.793   -0.140 0.734      0.857
  GHSL  embedding  998   18.361  13.820    0.339 0.732      0.858
  GHSL     median  998   18.785  14.125    0.386 0.720      0.848

===== Hanoi — same-source validation =====
target          predictor   n  RMSE_pp  MAE_pp  bias_pp     R2  pearson_r
  GHSL     median_localrf 895    6.325   4.376   -0.486  0.967      0.987
  GHSL  embedding_localrf 895    9.405   6.801   -0.683  0.927      0.969
  GHSL embedding_zeroshot 895   35.012  29.313  -23.210 -0.014      0.676
  GHSL    median_zeros

---

## Figures

In [5]:
# ── Figure · same-source agreement per run ───────────────────────────────────
# Panel titles name the metric and the city; the report supplies the figure
# title and caption.

matplotlib.rcParams.update({"font.size": 9, "axes.grid": True,
                            "grid.alpha": 0.3, "axes.axisbelow": True})

PANELS = [("R2", "R²  (higher is better)", None),
          ("RMSE_pp", "RMSE  (pp, lower is better)", None),
          ("bias_pp", "Bias  (pp, observed − predicted; + = under-predicts)", 0.0)]

for city in ("Milan", "Hanoi", "HCMC"):
    sub = metrics[metrics.city == city].copy()
    if not len(sub):
        continue
    sub["label"] = sub["target"] + " · " + sub["predictor"]
    sub = sub.sort_values("R2")
    fig, axes = plt.subplots(1, 3, figsize=(12, max(3, 0.32 * len(sub) + 1.6)),
                             sharey=True)
    for ax, (col, title, zero) in zip(axes, PANELS):
        ax.barh(sub["label"], sub[col], color="#4878a8")
        ax.set_title(title, fontsize=9)
        if zero is not None:
            ax.axvline(zero, color="#444444", lw=1.0)
        for y, v in enumerate(sub[col]):
            ax.text(v, y, f" {v:.2f}", va="center", fontsize=7.5)
    axes[0].set_ylabel(f"{city} · training target · predictor")
    fig.tight_layout()
    out = RESULTS / f"fig_{city.lower()}_samesource.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", out.name)

saved fig_milan_samesource.png


saved fig_hanoi_samesource.png
saved fig_hcmc_samesource.png


---

## Manifest

In [6]:
# ── Run manifest ─────────────────────────────────────────────────────────────
summary = dict(
    notebook="04_SameSource_Validation.ipynb",
    validation="same-source (vs the training target)",
    metrics=["RMSE_pp", "MAE_pp", "bias_pp", "R2", "pearson_r"],
    n_runs=len(RUNS),
    runs=[{"city": c, "target": t, "predictor": p, "n": int(len(DATA[(c, t, p)][0]))}
          for (c, t, p) in DATA],
)
with open(RESULTS / "samesource_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

print(f"\n-- wrote {len(list(RESULTS.iterdir()))} files to {RESULTS} --")
for f in sorted(RESULTS.iterdir()):
    print(f"   {f.name:46s} {f.stat().st_size / 1024:8.1f} kB")


-- wrote 5 files to D:\06_Polimi\2025-2026\IMD-Mapping\output\validation_samesource --
   fig_hanoi_samesource.png                           50.7 kB
   fig_hcmc_samesource.png                            51.2 kB
   fig_milan_samesource.png                           68.4 kB
   samesource_metrics.csv                              2.0 kB
   samesource_summary.json                             2.0 kB
